# 06 — PRIMERA (abstractive multi-documento)

**PRIMERA** (Xiao et al., ACL 2022) è un encoder-decoder Longformer (**LED**) pre-addestrato con un
obiettivo pensato apposta per la summarization **multi-documento** (Entity Pyramid: mascherare e
rigenerare frasi salienti condivise tra più documenti). Qui usiamo il checkpoint
**`allenai/PRIMERA-multinews`**, fine-tuned **proprio su Multi-News**.

Due differenze sostanziali rispetto a BART e PEGASUS (notebook 03/04):

1. **Input di 4096 token** (contro 1024): la maggior parte dei cluster di articoli del campione
   entra quasi per intero, invece di essere tagliata all'inizio del primo articolo.
2. **I confini tra articoli sono espliciti**: gli articoli (separati da `` ||||| `` nel dataset)
   vengono uniti dal token speciale `<doc-sep>`, che riceve **global attention** insieme al token
   iniziale `<s>` — è il meccanismo con cui il modello sa dove finisce un documento e inizia il
   successivo. Il troncamento assegna a ogni articolo lo **stesso budget di token** (la strategia
   del repo ufficiale [allenai/PRIMER](https://github.com/allenai/PRIMER)), così gli articoli in
   coda non vengono scartati in blocco come col troncamento semplice dei notebook 03/04.

⚠️ **Caveat di leakage** (identico a PEGASUS, notebook 04): il checkpoint è stato fine-tuned
sulla split *train* di questo dataset e il campione proviene da tutte e tre le split → i punteggi
sulle righe train sono ottimistici. Il confronto pulito è la media sulla sola split `test`
(vista dedicata nel notebook [05_confronto.ipynb](05_confronto.ipynb)).

Opera **solo sul campione condiviso**. Su CPU la generazione è molto lenta (beam search a 5 beam
su input 4096 → minuti per esempio): **consigliata la GPU** (pochi secondi a esempio). Riassunti
salvati incrementalmente in `results/summaries/primera_sample.tsv` con **ripresa**; metriche
ricalcolabili dai soli file salvati.

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer

In [1]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO     = 'primera'
SCOPE      = 'sample'    # questo notebook gira solo sul campione
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti
CHECKPOINT = 'allenai/PRIMERA-multinews'
MAX_INPUT  = 4096        # limite di input del modello (4x BART/PEGASUS)
MAX_LEN    = 1024        # lunghezza massima (token) del riassunto generato
NUM_BEAMS  = 5           # come l'esempio di valutazione del repo allenai/PRIMER

BASE   = su.trova_base_dir()
P      = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'
DEVICE = su.rileva_device()

print(f'Checkpoint : {CHECKPOINT}')
print(f'Device     : {DEVICE}')
print(f'Output     : {OUT_PATH}')

Checkpoint : allenai/PRIMERA-multinews
Device     : cuda
Output     : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\summaries\primera_sample.tsv


## Generazione dei riassunti

pyAutoSummarizer non supporta PRIMERA, quindi il modello viene usato direttamente via
`transformers` (`LEDForConditionalGeneration`), caricato **una sola volta** sul `DEVICE`
rilevato — la stessa prassi dei notebook 03/04. I parametri di generazione replicano l'esempio
di valutazione del repo ufficiale (`num_beams=5`, `max_length=1024`).

La preparazione dell'input è specifica di PRIMERA e per questo **non** usa
`su.prepara_documento` (che scioglierebbe il separatore `` ||||| `` in un newline, perdendo i
confini tra articoli — al ciclo condiviso passiamo `prepara=str.strip`):

1. il documento viene diviso in articoli sul separatore `` ||||| ``;
2. spazi e newline interni di ogni articolo vengono normalizzati a spazi singoli
   (come nel dataloader del repo PRIMER);
3. ogni articolo riceve lo stesso budget di token (`(4096 - 2) / n_articoli`) e viene troncato
   individualmente, poi gli articoli sono concatenati con `<doc-sep>`;
4. la `global_attention_mask` vale 1 su `<s>` e su ogni `<doc-sep>`, 0 altrove.

In [2]:
import re

import torch
from transformers import AutoTokenizer, LEDForConditionalGeneration

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
modello   = LEDForConditionalGeneration.from_pretrained(CHECKPOINT).to(DEVICE).eval()
DOCSEP_ID = tokenizer.convert_tokens_to_ids('<doc-sep>')

def costruisci_input(documento):
    """Token ids per PRIMERA: articoli troncati a pari budget, uniti da <doc-sep>."""
    articoli = [' '.join(a.split()) for a in re.split(r'\s*\|\|\|\|\|\s*', documento)]
    articoli = [a for a in articoli if a]
    budget = max((MAX_INPUT - 2) // max(len(articoli), 1), 8)  # riserva <s> ed </s>
    ids = []
    for articolo in articoli:
        # encode() aggiunge <s>...</s>: [1:-1] li toglie -> max budget-2 token di contenuto
        ids.extend(tokenizer.encode(articolo, truncation=True, max_length=budget)[1:-1])
        ids.append(DOCSEP_ID)
    ids = [tokenizer.bos_token_id] + ids[:MAX_INPUT - 2] + [tokenizer.eos_token_id]
    return torch.tensor([ids])

def genera(documento):
    input_ids = costruisci_input(documento).to(DEVICE)
    global_attention = torch.zeros_like(input_ids)
    global_attention[:, 0] = 1
    global_attention[input_ids == DOCSEP_ID] = 1
    with torch.no_grad():
        out = modello.generate(input_ids=input_ids,
                               global_attention_mask=global_attention,
                               use_cache=True, max_length=MAX_LEN, num_beams=NUM_BEAMS)
    return tokenizer.decode(out[0], skip_special_tokens=True)

esempi    = su.carica_campione(SAMPLE_PATH)
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT,
                                etichetta='PRIMERA ', prepara=str.strip)
scrittore.chiudi()

c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 587/587 [00:00<00:00, 31337.03it/s]
[transformers] Input ids are automatically padded from 476 to 512 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 1149 to 1536 to be a multiple of `config.attention_window`: 512


PRIMERA [1] media 6.7 s/esempio (saltati 3 gia' fatti)


[transformers] Input ids are automatically padded from 3028 to 3072 to be a multiple of `config.attention_window`: 512


PRIMERA [2] media 5.8 s/esempio (saltati 3 gia' fatti)


[transformers] Input ids are automatically padded from 835 to 1024 to be a multiple of `config.attention_window`: 512


...

PRIMERA [80] media 9.8 s/esempio (saltati 3 gia' fatti)


[transformers] Input ids are automatically padded from 2725 to 3072 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 1561 to 2048 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 788 to 1024 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 1457 to 1536 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 2053 to 2560 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 2507 to 2560 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 2116 to 2560 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 1380 to 1536 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded

PRIMERA [90] media 9.7 s/esempio (saltati 3 gia' fatti)


[transformers] Input ids are automatically padded from 1333 to 1536 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 1184 to 1536 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 3024 to 3072 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 1146 to 1536 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 4074 to 4096 to be a multiple of `config.attention_window`: 512
[transformers] Input ids are automatically padded from 3820 to 4096 to be a multiple of `config.attention_window`: 512


PRIMERA Completato: 97 nuovi, 3 saltati, 0 errori, 1066 s totali


## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/primera_sample_per_example.csv` e `…_aggregate.json` —
l'aggregato include le medie **per split** (fondamentale qui per il caveat di leakage sulla
split train).

In [3]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = su.carica_campione(SAMPLE_PATH)
config = {'checkpoint': CHECKPOINT, 'max_len': MAX_LEN, 'num_beams': NUM_BEAMS,
          'max_input_tokens': MAX_INPUT, 'device': DEVICE,
          'note': 'articoli uniti da <doc-sep> (global attention), budget di token uguale '
                  'per articolo; checkpoint multi_news (leakage su split train)',
          'libreria': 'transformers (parametri di generazione del repo allenai/PRIMER)'}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split (attenzione al leakage su train):')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

Metriche per-esempio : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\primera_sample_per_example.csv (100 righe)
Metriche aggregate   : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\primera_sample_aggregate.json
{
  "rouge1_f1": 0.4763809512839683,
  "rouge1_p": 0.4939050308340693,
  "rouge1_r": 0.4711228421359972,
  "rouge2_f1": 0.2296417177378229,
  "rouge2_p": 0.23882588782855307,
  "rouge2_r": 0.22834564309562316,
  "rougeL_f1": 0.26909819478364083,
  "rougeL_p": 0.28000654932595553,
  "rougeL_r": 0.26843234744268335,
  "bleu": 0.17955279534486673,
  "meteor": 0.5053769658778984,
  "parole_generate": 225.16
}

Medie per split (attenzione al leakage su train):
  test  (n=8): ROUGE-1 F1 = 0.441
  train (n=81): ROUGE-1 F1 = 0.483
  val   (n=11): ROUGE-1 F1 = 0.452


## Ispezione qualitativa

In [4]:
su.mostra_esempi(su.carica_campione(SAMPLE_PATH), riassunti, quanti=2)

--- row_id=425 (split=train) ---
GENERATO   : – Two days after the mass shooting at a Pittsburgh synagogue, a Republican candidate for a state Senate seat in Connecticut sent out a mailer that's being called anti-Semitic, the Washington Post reports. According to the Hartford Courant, the mailer shows Democratic state Rep. Matt Lesser "with large, beady eyes, holding fistfuls of hundred-dollar bills." On the other side, it reads, "Matt Lesser will take everything you worked for." The mailer was sent out by GOP candidate Ed Charamut. Lesser 
RIFERIMENTO: Just days after the slaying of 11 Jewish congregants at a Pittsburgh synagogue, a GOP candidate for a state Senate seat in Connecticut is accused of sending a mailer using an "age-old anti-Semitic trope." The ad sent out by Ed Charamut includes what the Washington Post calls a "money-grubbing" picture (here) of smiling opponent Matt Lesser, clutching $100 bills with a "crazed look in his eyes." Lesser says the original image of him was 